# CDT Paper Figure 6: Hi-C Validation

This notebook creates Figure 6 for the CDT paper:
- **(A)** Hi-C contact map concept showing enhancer-FNDC5 interaction
- **(B)** Mechanistic model of CTCF-mediated chromatin looping

**Note:** For publication, Panel A should ideally use actual Hi-C data from 3D Genome Browser.
This notebook creates a conceptual illustration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Arc, Circle
from matplotlib.lines import Line2D
import matplotlib.patheffects as path_effects

# Set style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

In [ ]:
def create_hic_conceptual_map(ax):
    """
    Create a conceptual Hi-C contact map showing the enhancer-FNDC5 interaction.
    This is a schematic representation - for publication, use actual Hi-C data.
    """
    # Create a simulated Hi-C-like matrix
    # Region: chr1:32.5Mb - 34.0Mb (1.5Mb window)
    n_bins = 150  # 10kb resolution
    
    # Base contact matrix (distance-dependent decay)
    contact_matrix = np.zeros((n_bins, n_bins))
    for i in range(n_bins):
        for j in range(n_bins):
            distance = abs(i - j)
            # Power law decay
            if distance == 0:
                contact_matrix[i, j] = 10
            else:
                contact_matrix[i, j] = 1.0 / (distance ** 0.8)
    
    # Add enhanced contact between enhancer and FNDC5
    # FNDC5: chr1:32.87Mb -> bin ~37 (from 32.5Mb)
    # Enhancer: chr1:33.59Mb -> bin ~109
    fndc5_bin = 37
    enhancer_bin = 109
    
    # Add strong off-diagonal contact (loop)
    loop_strength = 3.0
    for di in range(-3, 4):
        for dj in range(-3, 4):
            dist = np.sqrt(di**2 + dj**2)
            if dist < 4:
                i, j = fndc5_bin + di, enhancer_bin + dj
                if 0 <= i < n_bins and 0 <= j < n_bins:
                    contact_matrix[i, j] += loop_strength * np.exp(-dist/2)
                    contact_matrix[j, i] += loop_strength * np.exp(-dist/2)
    
    # Add TAD structure
    tad_boundaries = [25, 75, 125]
    for boundary in tad_boundaries:
        if boundary < n_bins:
            # Slightly reduce contacts across TAD boundaries
            for i in range(n_bins):
                for j in range(n_bins):
                    if (i < boundary and j > boundary) or (i > boundary and j < boundary):
                        contact_matrix[i, j] *= 0.85
    
    # Log transform for visualization
    contact_matrix = np.log1p(contact_matrix)
    
    # Plot
    im = ax.imshow(contact_matrix, cmap='YlOrRd', aspect='equal', origin='lower')
    
    # Mark the enhancer-FNDC5 contact
    circle = Circle((enhancer_bin, fndc5_bin), 6, fill=False, color='blue', linewidth=2, linestyle='--')
    ax.add_patch(circle)
    
    # Add annotations
    ax.annotate('Enhancer-FNDC5\ncontact', xy=(enhancer_bin, fndc5_bin), 
                xytext=(enhancer_bin + 20, fndc5_bin - 20),
                fontsize=9, color='blue',
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
    
    # Axis labels
    tick_positions = [0, 37, 75, 109, 149]
    tick_labels = ['32.5', '32.87\n(FNDC5)', '33.25', '33.59\n(Enhancer)', '34.0']
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=8)
    
    ax.set_xlabel('Chromosome 1 position (Mb)', fontsize=10)
    ax.set_ylabel('Chromosome 1 position (Mb)', fontsize=10)
    ax.set_title('K562 Hi-C Contact Map (Conceptual)', fontsize=11, fontweight='bold')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.6)
    cbar.set_label('Contact frequency (log)', fontsize=9)
    
    return ax

In [ ]:
def create_chromatin_loop_model(ax):
    """
    Create a schematic of CTCF-mediated chromatin looping.
    """
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Title
    ax.text(5, 7.5, 'CTCF-Mediated Chromatin Looping Model', 
            ha='center', va='top', fontsize=11, fontweight='bold')
    
    # Draw chromatin fiber as a curved line
    # Linear representation at top
    ax.annotate('', xy=(9, 6.5), xytext=(1, 6.5),
                arrowprops=dict(arrowstyle='-', color='gray', lw=3))
    
    # Mark positions on linear DNA
    # FNDC5 at left
    ax.plot([2], [6.5], 's', color='#2E86AB', markersize=12)
    ax.text(2, 7.0, 'FNDC5', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#2E86AB')
    ax.text(2, 6.1, 'chr1:32.87Mb', ha='center', va='top', fontsize=7, color='gray')
    
    # CTCF site (= Gradient peak at -56.7kb)
    ax.plot([5], [6.5], 'D', color='#E94F37', markersize=12)
    ax.text(5, 7.0, 'CTCF', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#E94F37')
    ax.text(5, 6.0, 'Gradient peak', ha='center', va='top', fontsize=8, fontweight='bold', color='#E94F37')
    ax.text(5, 5.6, '(-56.7kb)', ha='center', va='top', fontsize=8, color='#E94F37')
    
    # Enhancer at right
    ax.plot([7.5], [6.5], 'o', color='#44AF69', markersize=12)
    ax.text(7.5, 7.0, 'Enhancer', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#44AF69')
    ax.text(7.5, 6.1, 'chr1:33.59Mb', ha='center', va='top', fontsize=7, color='gray')
    
    # Distance annotation
    ax.annotate('', xy=(7.3, 5.2), xytext=(2.2, 5.2),
                arrowprops=dict(arrowstyle='<->', color='black', lw=1))
    ax.text(4.75, 5.0, '726 kb linear distance', ha='center', va='top', fontsize=8)
    
    # Arrow pointing down
    ax.annotate('', xy=(5, 4.3), xytext=(5, 4.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(5.5, 4.5, 'CTCF-mediated\nlooping', ha='left', va='center', fontsize=8, style='italic')
    
    # Draw looped chromatin structure
    # Create a loop shape
    theta = np.linspace(0, np.pi, 50)
    loop_x = 5 + 2.5 * np.cos(theta)
    loop_y = 2.5 + 1.5 * np.sin(theta)
    ax.plot(loop_x, loop_y, '-', color='gray', lw=3)
    
    # Add the remaining linear parts
    ax.plot([1, 2.5], [2.5, 2.5], '-', color='gray', lw=3)
    ax.plot([7.5, 9], [2.5, 2.5], '-', color='gray', lw=3)
    
    # Mark positions on looped structure
    # FNDC5 (now at base of loop, left side)
    ax.plot([2.5], [2.5], 's', color='#2E86AB', markersize=12)
    ax.text(2.5, 2.0, 'FNDC5', ha='center', va='top', fontsize=9, fontweight='bold', color='#2E86AB')
    
    # CTCF at top of loop (Gradient peak)
    ax.plot([5], [4.0], 'D', color='#E94F37', markersize=12)
    ax.text(5.6, 4.0, 'CTCF\n(Gradient peak)', ha='left', va='center', fontsize=8, fontweight='bold', color='#E94F37')
    
    # Enhancer (brought close to FNDC5 by loop)
    ax.plot([7.5], [2.5], 'o', color='#44AF69', markersize=12)
    ax.text(7.5, 2.0, 'Enhancer', ha='center', va='top', fontsize=9, fontweight='bold', color='#44AF69')
    
    # Show spatial proximity
    ax.annotate('', xy=(7.3, 2.7), xytext=(2.7, 2.7),
                arrowprops=dict(arrowstyle='<->', color='#9B59B6', lw=2, linestyle='--'))
    ax.text(5, 3.0, 'Spatial proximity\n(3D contact)', ha='center', va='bottom', 
            fontsize=8, color='#9B59B6', fontweight='bold')
    
    # Add regulatory arrow
    ax.annotate('', xy=(3.0, 2.3), xytext=(7.0, 2.3),
                arrowprops=dict(arrowstyle='->', color='#44AF69', lw=2))
    ax.text(5, 1.7, 'Regulatory\ninteraction', ha='center', va='top', 
            fontsize=8, color='#44AF69', style='italic')
    
    # Legend
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#2E86AB', markersize=10, label='Target gene (FNDC5)'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='#E94F37', markersize=8, label='CTCF site (Gradient peak)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#44AF69', markersize=10, label='Enhancer'),
    ]
    ax.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=8, 
              frameon=True, bbox_to_anchor=(0.5, -0.05))
    
    # A compartment annotation
    ax.add_patch(FancyBboxPatch((0.5, 1.3), 8.5, 3.5, 
                                 boxstyle="round,pad=0.1", 
                                 facecolor='#E8F5E9', edgecolor='#81C784', 
                                 alpha=0.3, lw=2, zorder=0))
    ax.text(9.2, 3.0, 'A compartment\n(active)', ha='left', va='center', 
            fontsize=8, color='#388E3C', rotation=0)
    
    return ax

In [ ]:
# Create Figure 6
fig = plt.figure(figsize=(14, 6))

# Panel A: Hi-C conceptual map
ax1 = fig.add_subplot(121)
create_hic_conceptual_map(ax1)
ax1.text(-0.1, 1.05, 'A', transform=ax1.transAxes, fontsize=16, fontweight='bold', va='top')

# Panel B: Chromatin loop model
ax2 = fig.add_subplot(122)
create_chromatin_loop_model(ax2)
ax2.text(-0.05, 1.05, 'B', transform=ax2.transAxes, fontsize=16, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig('figure6_hic_validation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nFigure 6 saved as 'figure6_hic_validation.png'")
print("\nNote: Panel A is a conceptual illustration.")
print("For publication, consider using actual Hi-C data from 3D Genome Browser:")
print("  - URL: http://3dgenome.fsm.northwestern.edu/view.php")
print("  - Cell type: K562")
print("  - Region: chr1:32,500,000-34,000,000")

In [ ]:
# Download the figure
from google.colab import files
files.download('figure6_hic_validation.png')